In [ ]:
#========================================================================
# Name: plot_fig_S1_mie_scattering.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
#
# Description: Demonstrate Mie scattering theory for cloud water droplets
#              and mass to reflectivity relationships.
#========================================================================

import numpy as np
import matplotlib.pyplot as plt
import miepython

import warnings
warnings.filterwarnings("ignore")

In [2]:
def compute_Q_vectorized(m, radii, wavelength):
    """
    Compute extinction, scattering, and absorption efficiencies using Mie theory.
    This vectorized version accepts an array of radii for fast computation.
    
    Compatible with miepython ≥3.4 (requires wavelength as third argument).
    """
    # If radii is an array, x will be an array.
    #x = 2.0 * np.pi * radii / wavelength
    diameters = 2.0 * radii

    # miepython is vectorized: it accepts an array of size parameters (x)
    # and returns arrays of efficiencies.
    qext, qsca, qback, g = miepython.efficiencies(m, diameters, wavelength)
    
    qabs = qext - qsca
    return qext, qsca, qabs


def maxwell_garnett(m_incl, m_host, f_incl):
    """
    Maxwell-Garnett effective permittivity (inclusions in host).
    m_incl, m_host : complex refractive indices
    f_incl : volume fraction of inclusions
    returns: m_eff (complex)
    """
    eps_incl = m_incl**2
    eps_host = m_host**2
    num = eps_incl + 2*eps_host + 2*f_incl*(eps_incl - eps_host)
    den = eps_incl + 2*eps_host - f_incl*(eps_incl - eps_host)
    eps_eff = eps_host * (num / den)
    m_eff = np.sqrt(eps_eff)
    return m_eff


# -------------------------------------------------------
# Example optical properties from Querry & Hale (1978) and Warren (1984)
# -------------------------------------------------------
# Wavelengths in meters
wl_VIS = 0.64    # 0.55 μm (visible)
wl_IR  = 11.2    # 11.2 μm (infrared)

# Refractive indices
#m_liq_VIS = 1.333 + 1.96e-9j
m_liq_VIS = 1.331 + 1.6e-8j
m_liq_IR  = 1.153 + 0.0968j

#m_ice_VIS = 1.3110 + 3.11e-9j
m_ice_VIS = 1.3083 + 1.22e-8j
m_ice_IR  = 1.0925 + 0.248j

m_air = 1.0 + 0.0j

#m_snow_VIS = 1.25 + 3.11e-9j
#m_snow_IR  = 1.05 + 0.15j

rho_ice = 890.0
rho_snow = 100.0
f_ice = rho_snow / rho_ice   # volume fraction of ice in snow

m_snow_mg_vis = maxwell_garnett(m_ice_VIS, m_air, f_ice)
m_snow_mg_ir  = maxwell_garnett(m_ice_IR,  m_air, f_ice)

m_snow_VIS = m_snow_mg_vis
m_snow_IR = m_snow_mg_ir

radii = np.logspace(-6, -3, 100)*1.e6 

# -------------------------------------------------------
# Compute efficiencies (The new, fast way)
# -------------------------------------------------------
print("Starting vectorized computation...")
start_time = time.time()

# Define the cases to run in a list of tuples for easy iteration
cases_to_run = [
    ('Liquid VIS', m_liq_VIS, wl_VIS),
    ('Liquid IR',  m_liq_IR,  wl_IR),
    ('Ice VIS',    m_ice_VIS, wl_VIS),
    ('Ice IR',     m_ice_IR,  wl_IR),
    ('Snow VIS',   m_snow_VIS, wl_VIS),
    ('Snow IR',    m_snow_IR, wl_IR)
]

cases = {}
for name, m, wl in cases_to_run:
    print('Running:', name)
    # A single call computes efficiencies for ALL radii
    qext, qsca, qabs = compute_Q_vectorized(m, radii, wl)
    cases[name] = (qext, qsca, qabs)

end_time = time.time()
print(f"\nVectorized computation finished in {end_time - start_time:.4f} seconds.")

# Save the dictionary to a pickle file
#with open("mie_calcs.pkl", "wb") as file:
#    pickle.dump(cases, file)

Starting vectorized computation...
Running: Liquid VIS
Running: Liquid IR
Running: Ice VIS
Running: Ice IR
Running: Snow VIS
Running: Snow IR

Vectorized computation finished in 1.0400 seconds.


In [2]:
with open('mie_calcs.pkl', 'rb') as file:
    cases = pickle.load(file)
#radii = np.array([1,5,10,20,50,100]) # 200 points instead of 6
radii = np.logspace(-6, -3, 100)*1.e6 # 200 points instead of 6

In [3]:
plt.rcParams['text.usetex'] = True

In [24]:
# -------------------------------------------------------
# Plot
# -------------------------------------------------------
fig = plt.figure(figsize=(12,5),constrained_layout=True)
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)
axlist = [ax1,ax2]
Fontsize=14
if True:
    for name, (qext, qsca, qabs) in cases.items():
        if name == 'Liquid VIS':
            color = 'red'
        elif name == 'Liquid IR':
            color = 'salmon'
        elif name == 'Ice VIS':
            color = 'blue'
        elif name == 'Ice IR':
            color = 'deepskyblue'
        elif name == 'Snow VIS':
            color = 'darkgreen'
        elif name == 'Snow IR':
            color = 'mediumseagreen'
    
        # qsca and qabs are now arrays, perfect for plotting directly
        ax1.plot(radii, qsca, label=f"{name} (scattering)",c=color)
        ax2.plot(radii, qsca, label=f"{name} (scattering)",c=color)
        ax1.plot(radii, qabs, '--', label=f"{name} (absorption)",c=color)
        ax2.plot(radii, qabs, '--', label=f"{name} (absorption)",c=color)

for ax in axlist:
    ax.set_xlabel("Radius [$\\mu$m]",fontsize=Fontsize)
    ax.set_ylabel("Efficiency $Q$",fontsize=Fontsize)
    ax.grid(which='both',lw=1,c='dimgrey',ls='dotted')
    ax.set_xscale('log')
    ax.tick_params(labelsize=Fontsize)
#plt.title("Mie Efficiencies for Different Hydrometeor Types and Wavelengths")
ax1.set_ylim(1e-8,1.e1) # Set a reasonable y-limit for better visualization
ax2.set_ylim(0,2.5) # Set a reasonable y-limit for better visualization
ax1.set_yscale('log')
ax1.legend(fontsize=8, loc='center',ncol=2,bbox_to_anchor=(0.68,0.65))

ax1.text(1.,0.99,'(a)',fontsize=Fontsize*2.5,ha='right',va='top',transform=ax1.transAxes)
ax2.text(1.,0.99,'(b)',fontsize=Fontsize*2.5,ha='right',va='top',transform=ax2.transAxes)

for ax in axlist:
    ax.set_xlim(1,1.e3)
#plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make room for legend

save_path = '/global/homes/m/mckenna/figures/jgra_paper/'
file_name = 'mie_scattering.png'
plt.savefig(save_path+file_name,dpi=300)
#plt.show()
plt.close()

In [9]:
#np.max(radii)

In [10]:
#np.logspace(-6, -2, 100)